In [1]:
import pandas as pd
import numpy as np
import os

# Metrics Table

In [26]:
BASE_PATH = "/hdd/ivny/results/"
DATASETS = ["trivia_qa", "mmlu", "squadv2"]

# Map base method name → paper name
BASE_METHOD_MAP = {
    "lnll": "Token Probability",
    "lnll_gen": "Token Probability",
    "p_true_mc": "Self-Evaluation",
    "semantic_uncertainty": "Semantic Uncertainty",
    "linguistic_confidence": "Linguistic Confidence",
}


for DATASET in DATASETS:
    rows = []
    dataset_path = os.path.join(BASE_PATH, DATASET)

    for conf_mode in os.listdir(dataset_path):
        conf_path = os.path.join(dataset_path, conf_mode)
        if not os.path.isdir(conf_path):
            continue

        # -----------------------------
        # Detect scalar vs dist
        # -----------------------------
        is_dist = conf_mode.startswith("dist_")
        base_mode = conf_mode.replace("dist_", "")

        if base_mode not in BASE_METHOD_MAP:
            continue

        estimation_method = BASE_METHOD_MAP[base_mode]

        for model_family in os.listdir(conf_path):
            family_path = os.path.join(conf_path, model_family)
            if not os.path.isdir(family_path):
                continue

            for model_name in os.listdir(family_path):
                model_path = os.path.join(family_path, model_name)
                if not os.path.isdir(model_path):
                    continue

                timestamps = sorted(os.listdir(model_path))
                if not timestamps:
                    continue

                timestamp = timestamps[-1]
                metrics_path = os.path.join(
                    model_path, timestamp, "eval_metrics_summary.csv"
                )

                if not os.path.exists(metrics_path):
                    continue

                df = pd.read_csv(metrics_path, index_col=0).transpose()

                row = {
                    "Model": model_name,
                    "Estimation Method": estimation_method,
                    "Acc": None,
                    "Acc_dist": None,
                    "ECE": None,
                    "dECE": None,
                    "dECE_pt": None,
                    "AUROC": None,
                    "dAUROC": None,
                    "dAUROC_pt": None,
                }

                # -----------------------------
                # Fill metrics
                # -----------------------------
                if not is_dist:
                    row["Acc"] = df["accuracy_scalar_with_abstention"].iloc[0]
                    row["ECE"] = df["ece_scalar"].iloc[0]
                    row["AUROC"] = df["auroc_scalar"].iloc[0]

                else:
                    row["Acc_dist"] = df["accuracy_scalar_with_abstention"].iloc[0]
                    row["dECE"] = df["dECE_equal_width"].iloc[0]
                    row["dECE_pt"] = df["dECE_scalar"].iloc[0]
                    row["dAUROC"] = df["dAUROC"].iloc[0]
                    row["dAUROC_pt"] = df["dAUROC_scalar"].iloc[0]

                rows.append(row)

    # =========================
    # Merge scalar + dist rows
    # =========================
    results_df = (
        pd.DataFrame(rows)
        .groupby(["Model", "Estimation Method"], as_index=False)
        .first()
    )

    results_df.sort_values(
        by=["Model", "Estimation Method"],
        inplace=True
    )

    results_df.reset_index(drop=True, inplace=True)

    results_df["Model"] = results_df["Model"].str.upper()
    results_df.to_csv(f"metrics_table_{DATASET}.csv", index=True)
    results_df

# Post-Hoc Calibration Table

In [ ]:
BASE_PATH = "/hdd/ivny/results/"
DATASETS = ["trivia_qa", "mmlu", "squadv2"]

# Map base method name → paper name
BASE_METHOD_MAP = {
    "lnll": "Token Probability",
    "lnll_gen": "Token Probability",
    "p_true_mc": "Self-Evaluation",
    "semantic_uncertainty": "Semantic Uncertainty",
    "linguistic_confidence": "Linguistic Confidence",
}


for DATASET in DATASETS:
    rows = []
    dataset_path = os.path.join(BASE_PATH, DATASET)

    for conf_mode in os.listdir(dataset_path):
        conf_path = os.path.join(dataset_path, conf_mode)
        if not os.path.isdir(conf_path):
            continue

        # -----------------------------
        # Detect scalar vs dist
        # -----------------------------
        is_dist = conf_mode.startswith("dist_")
        base_mode = conf_mode.replace("dist_", "")

        if not is_dist:
            continue

        if base_mode not in BASE_METHOD_MAP:
            continue

        estimation_method = BASE_METHOD_MAP[base_mode]

        for model_family in os.listdir(conf_path):
            family_path = os.path.join(conf_path, model_family)
            if not os.path.isdir(family_path):
                continue

            for model_name in os.listdir(family_path):
                model_path = os.path.join(family_path, model_name)
                if not os.path.isdir(model_path):
                    continue

                timestamps = sorted(os.listdir(model_path))
                if not timestamps:
                    continue

                timestamp = timestamps[-1]
                metrics_path = os.path.join(
                    model_path, timestamp, "eval_metrics_summary.csv"
                )

                if not os.path.exists(metrics_path):
                    continue

                df = pd.read_csv(metrics_path, index_col=0).transpose()

                row = {
                    "Model": model_name,
                    "Estimation Method": estimation_method,
                    "raw_dECE": None,
                    "raw_dECE_pt": None,
                    "dECE_reg": None,
                    "dECE_pt_reg": None,
                    "dECE_kl": None,
                    "dECE_pt_kl": None,
                    "raw_dAUROC": None,
                    "raw_dAUROC_pt": None,
                    "dAUROC_reg": None,
                    "dAUROC_pt_reg": None,
                    "dAUROC_kl": None,
                    "dAUROC_pt_kl": None,
                }

                # -----------------------------
                # Fill metrics
                # -----------------------------
                
                row["Acc_dist"] = df["accuracy_scalar_with_abstention"].iloc[0]
                row["dECE"] = df["dECE_equal_width"].iloc[0]
                row["dECE_pt"] = df["dECE_scalar"].iloc[0]
                row["dAUROC"] = df["dAUROC"].iloc[0]
                row["dAUROC_pt"] = df["dAUROC_scalar"].iloc[0]

                rows.append(row)

    # =========================
    # Merge scalar + dist rows
    # =========================
    results_df = (
        pd.DataFrame(rows)
        .groupby(["Model", "Estimation Method"], as_index=False)
        .first()
    )

    results_df.sort_values(
        by=["Model", "Estimation Method"],
        inplace=True
    )

    results_df.reset_index(drop=True, inplace=True)

    results_df["Model"] = results_df["Model"].str.upper()
    results_df.to_csv(f"metrics_table_{DATASET}.csv", index=True)
    results_df